输入特征多

图片是由像素构成，现在的手机拍摄的相片都有几千万像素。黑白图片每个像素只有一个灰度值。彩色图片每个像素有（R,G,B）三个值，分别代表红、绿、蓝。 如果一个图片长宽为1000x1000个像素的彩色图片，如果我们构造一个神经网络来处理这个图片，它的输入特征有300万个，如果隐藏层为1000个神经元，输出层为1个神经元，那么这个网络将有30亿个参数，以Float32来存储，模型大小为12GB。这么大的模型推理一次的运算代价是非常大的。

局部性

在图像里局部区域内的像素构成了重要的特征，每个像素和它周围局部区域内的像素关系紧密。比如我们识别一只猫，只要观察这个猫在图片里的区域就可以确定，无需依赖其他地方的像素信息。如果我们用全连接神经网络，每个神经元都是和所有输入像素相连的。隐藏层的神经元判断一个特征，只需要局部像素就可以，没有必要用全部的像素。这些没有必要的连接浪费了大量的权重参数。

平移不变性

不论一只猫在图片的左上角还是右下角。都不改变它是一只猫的语义信息。在计算机视觉中，语义信息（semantic information）是指图像中所表达的高层次、具有人类可理解意义的内容。如果我们训练了一个神经元，它可以接入局部的像素信息，识别一个猫，那么它就可以被重复使用，识别图片任意区域的猫。

步兵扫雷

我在学习卷积神经网络时，想到一个很好的类比，那就是步兵扫雷，它和我们在视觉问题里遇到的问题非常类似。

输入特征多 雷区可能很大，你不能造一个和雷区一样大的探测器，那样成本太大。

局部性 一个地雷只在一个雷区的局部。探测一个地雷，只需要探测一个局部地区就可以确定。

平移不变性 不论一个地雷在雷区的哪个位置，都是一样的。地雷的属性不会改变。

那现实中步兵是如何探雷的呢？

假设雷区是一个6x6的区域，而探雷器能探测的区域为3x3，那么步兵会用探雷器在整个雷区做从上到下，从左到右的扫描，来检查每一个3x3的区域是否有地雷。

我们来正式学习卷积核，它的原理和我们上一节讲的探雷器非常相似。

首先我们来定义输入和参数。

输入 假设输入是一个照片，它的长和宽都是6个像素。如果它是彩色照片，那么它需要用一个6x6x3的tensor来表示。最后一个维度的3，表示RGB，3个通道（channel）。如果是黑白照片，那么它需要用一个6x6x1的tensor来表示，最后一个维度是1，代表只有一个通道：灰度值通道。下边我们首先以6x6的黑白照片为例进行讲解。

参数 一个3x3的卷积操作一共有10个参数，包括一个3x3的卷积核参数，还有一个偏置b参数。卷积核和偏置一起被称为一个过滤器（Filter）。

在进行卷积运算时，卷积核从图像的左上角开始，从上到下，从左到右，进行扫描。每次对3x3的像素进行卷积运算。卷积运算对每个位置的像素值和卷积核对应位置的权重相乘，然后累加，最后再加上偏置b，得到z值，然后z值经过激活函数后，得到a值。a值为新的“图片”上的一个“像素”。新的“图片”，我们称为特征图，特征图上的每个“像素”都是从原始图片上提取的一个特征。比如在检测猫的任务中，一个a值可能代表是否在这个卷积区域发现了猫的眼睛。

当提取了原始图片的第一个3x3区域的特征后，卷积操作就向右滑动一个像素，然后用同样的卷积参数对下一个区域进行特征提取。

卷积操作完美解决了视觉问题的三个痛点。

图片输入特征多 图片输入特征多，但是一个3x3的卷积操作只有10个参数，就可以对整个图片进行扫描。

特征局部性 卷积操作的每个运算只在特定相邻区域内进行，并不要所有输入特征都参与运算。

平移不变性 卷积操作在整个图片上进行滑动检测，就是假设图片的特征具有平移不变性。

所以对于一个3x3的卷积操作，它只用10个参数就可以在任意大的图片上提取特征。

卷积操作和全连接操作并没有本质的不同。对于一个3x3的卷积操作，你可以看成是一个连接9个输入的神经元，它有9个weight参数，一个bias参数。这个神经元会用同样的参数对输入特征图上的任意3x3区域进行计算，形成输出的特征图。

所以卷积操作可看作带有先验知识（局部性、平移不变性）的参数共享全连接层。

上边我们讲的是单通道的黑白照片作为输入，那么对于3通道RGB彩色图片作为输入，卷积操作会有什么不同呢？

如果输入特征图是3通道的，那么卷积核也必须是3通道的，卷积核的shape为3x3x3，偏置还是只有一个参数b，一共28个参数。在进行卷积操作时，第一个通道的输入和第一个通道的卷积核参数进行卷积操作，第二个通道的输入和第二个通道的卷积核参数进行卷积操作，第三个通道也是类似，然后将三个通道得到的卷积值相加，再加上偏置值b，最终一个三通道卷积操作还是得到一个值。最终输出的通道数还是1个。换句话说，就是对应通道，对应高，对应宽位置上的输入和权重相乘，最后累加所有的值，再加上b值，经过激活函数，得到3x3x3输入区域的特征值a。

多Filter操作

上边我们讲过如果输入特征图是多通道的，如果通道数为C，那么卷积核的通道数也必须为C，但是输出的特征图还只是一个通道。一个卷积操作可能是检测图片里某一种特征，比如在检测猫的例子里，是检测猫的眼睛，我们还需要检测其他特征，比如猫的耳朵，猫的嘴巴等器官。那么我们在一个卷积层里可以定义多个Filter（卷积操作），当我们定义了几个Filter，输出的通道数就为几。

比如上图中，我们对一个输入通道为3的特征图进行卷积，定义了两个Filter，每个Filter的卷积核都是3x3x3。因为定义了2个Filter，最终得到的输出特征图的通道数为2。

定义多个卷积层

之前我们讲过，深度神经网络为什么效果好，是因为每一层都可以看做是一个特征提取器，浅层提取初级特征，深层在初级特征的基础上提取高级的特征。

同样卷积层也可以定义多个进行累加。因为输入的是原始的图像的RGB特征图，第一层，我们定义多个Filter，它们检测原始图像中的点、线、纹理等，生成输出的特征图。输出特征图仍然具备视觉问题的特点，即局部性和平移不变性。所以可以在输出的特征图上继续定义卷积层，也可以定义多个Filter，用来从点、线、纹理这样的基础特征上提取眼睛、鼻子、嘴巴等这样的高级特征。需要注意的是每一个卷积层上的卷积核的通道数，都必须等于它输入特征图的通道数。

所以一般在定义卷积神经网络时，会堆叠多个卷积层，每个卷积层也会定义多个Filter，来提取不同的特征。

同理你可以设计出横条纹检测的卷积核、斜条纹检测的卷积核。

但是需要注意的是，训练神经网络时，我们并不需要手动设计卷积核的权重，在训练过程中它会自己学到检测各种特征的卷积核。

PyTorch 里的特征图的表示

在我们前面的介绍中，特征图的格式是按照“高 × 宽 × 通道数”（Height × Width × Channels，HWC）来组织的。在一些深度学习框架中，比如 TensorFlow，默认也使用这种格式（channels_last）。 但是在 PyTorch 中，特征图的张量（tensor）采用的是“通道数 × 高 × 宽”（Channels × Height × Width，CHW）这种格式，也就是 channels_first。 比如，对于一张高为 224、宽为 112 的 3 通道彩色图片，在读入 PyTorch 后，它的张量形状为：[3, 224, 112]。 由于一般学术论文里默认采用HWC格式，所以本书后边都采用HWC格式。但是你在调试PyTorch代码时需要注意格式的不同。

10.3

卷积操作的进阶

感受野

对原始的图片连续做两次的3x3的卷积操作。

如果原始图片的尺寸为6x6，那么做完第一次3x3的卷积后，生成的特征图为4x4，在这4x4的特征图里，每个特征都是由原始图片里3x3的像素参与运算得到。所以我们说第一次卷积操作生成的特征图每个特征的感受野是3x3。

在第一次卷积生成的特征图上我们继续做第二次卷积，得到的特征图大小为2x2，在这个2x2的特征图里，每个特征都是由原始图片里5x5的像素参与运算得到的。所以我们说第二次卷积操作生成的特征图每个特征的感受野是5x5。

感受野是指当前特征图里的每个特征是由原始多大区域的像素参与运算得到的。

我们通过观察可以看到，随着卷积的不断进行，生成的特征图越来越小，但是特征图的感受野越来越大。这刚好也契合了深度神经网络的特性，浅层网络发现细节局部的特征，深层网络发现更加宏观整体的特征。

卷积核的大小

之前我们讲的卷积核的大小为3x3，当然你也可以设置其他大小，比如5x5,7x7等，但是一般我们都设为奇数的正方形。目前最常用的卷积核都是3x3的。这是因为奇数核具有唯一的中心像素，便于定位特征。例如，3×3核的中心是第2行第2列，这为卷积操作提供了对称的参考点，使特征提取更直观。

你不用担心3x3的卷积核能看到的像素太少，以至于发现不了什么有用的特征。上边我们通过分析特征图的感受野知道，经过几次3x3的卷积，后边特征图的感受野会不断增大的。

步长

步长（stride），之前我们讲卷积核在特征图上滑动，不论从左到右，还是从上到下都是一个像素一个像素的滑动，也就是步长为1。你也可以设置步长为其他值，比如2，那么卷积操作生成的特征图将更快的缩小。一般情况下，我们都设置步长为1，这样能提取更多细节特征。

如果你的卷积核设置比较大，比如7x7，证明你要检测的特征比较大，那么你可以对应的将步长设置为2。这样每次滑动后，滑过的区域相对于检测框来说不大，漏掉要检测的特征概率较小。 如果你的卷积核设置比较小，比如3x3，那么你要检测的特征也比较小，那么你最好把步长设置为1。如果设置为2的话，可能会漏掉你要检测的特征。

 Padding
 
我们目前学到的卷积操作有两个问题：

每次经过卷积操作，得到新的特征图会变小，在图片原本尺寸就很小的情况下，特征图经过几层就变的非常小，无法进行卷积操作了。我们希望有一种方法能保持特征图的尺寸，从而可以添加更多的卷积层，增加网络的深度。

卷积操作，对于处于图片中央的像素，会被多次用于计算，而处于边缘的像素用于计算的次数则较少。 比如对于处于图片中央的像素，在3x3的卷积核下，会参与9次运算：

但是，如果一个像素处于左上角，它只参与一次运算：

为了解决上边的两个问题，人们引入了Padding操作。

Padding操作就是在图片周围添加全0的像素。相当于给原始图片周围加了一圈黑边。当然也有其他Padding的做法，比如有一种填充方法就是重复边缘的像素来进行填充。我们最常用的还是全零填充。

以0填充为例，Padding为1，就是在图像周围填充一圈0。Padding为2，就是在图像周围填充两圈0。

Size 为1的Padding后的6x6的图像变为了8x8。它解决了上边所说的两个问题，原始图片左上角的像素可以参与4次计算。另外在填充后8x8的图片上按照步长为1,3x3的卷积，生成的特征图，还是6x6。它保持了和原始实际图片同样的大小。如果我们还要接着进行卷积操作，同时还想保证特征图的尺寸不变，那么我们可以在生成的6x6的特征图上继续进行填充。

10.4

在一个完整的卷积神经网络里，除了之前我们介绍的卷积层外，还有一种重要的网络结构就是池化层。

对于一个4x4的特征图，池化窗口大小（或者称为池化核大小）为2x2，步长为2，它会在每个池化窗口内取当前特征图的最大值，作为输出，生成输出特征图。

池化层一般跟在一个或者多个卷积层的后边，构成一个特征提取模块。一个卷积神经网络里可以有多个这样的模块。池化层的作用有以下几个：

减少特征图的大小。

过滤一些干扰信息，因为最大池化层只提取一个区域内的最强激活值，忽略其他值。所以它有排除干扰信息的作用。

提供小范围的平移不变性，因为不论最大特征出现在池化窗口内的任意位置，都以最大特征作为输出。

当然你可以改变池化窗口的大小，但一般我们设置为2x2。你也可以改变池化操作的步长，但一般情况下，我们设置为和池化窗口一样的大小，这样每次池化操作就没有重叠，可以起到更好的减少特征图尺寸。另外池化层一般也不设置Padding，因为使用池化层的目的就是为了减少特征图尺寸并保留最强特征。

所以一般在卷积神经网络里最常见的最大池化层，size为2x2，步长为2，没有Padding，它输出的特征图的长和宽都是输入特征图的一半。

多通道池化层

多通道池化层是分通道进行的。在每个通道上独立进行池化操作。所以池化操作并不改变输入的通道数。这点和卷积操作不同，一个卷积核，它的通道数必须和输入通道数一样，计算时对多通道数据进行融合计算，最终不论输入是几通道，输出只有一个通道。为了输出多通道，就要设置多个卷积核。 为什么它们两者会有这种区别呢？因为卷积操作有可学习的参数，假如输入通道有两个，一个通道提取的是颜色特征，一个通道提取的形状特征，这个卷积核会融合颜色和形状特征，产生出一个新的特征，比如说是苹果这个类别的特征。但是池化层，以最大池化层为例，它并没有可学习的参数，它也没有对多通道特征进行融合的能力，它只是保留区域内最大特征，消除其他特征。并且不同通道的特征没有可比性，在不同通道的特征里取一个最大值没有意义。

 和卷积层相比
卷积层如果设置步长不为1，不加Padding，也可以减少输出特征图的尺寸。那么和池化层相比两者有什么不同呢？答案是卷积层有可学习的参数，而池化层没有需要学习的参数，运算效率更高。 卷积层的作用是利用多通道的信息，生成新的特征。而最大池化层只是按照规则保留区域内最大特征，消除其他特征。

10.4.6 平均池化操作
除了最大池化外，平均池化是另一种常见的池化操作方式。与最大池化不同，平均池化会对池化窗口内的所有特征值取平均值，并将该平均值作为输出特征图的对应位置值。

最大池化操作聚焦关键特征，适合做图像分类这样的任务。 平均池化对局部特征进行平滑处理，适合对特征图进行整体浓缩，但不强调单点极值，常用于去噪、建模全局信息。最常用的是全局平均池化层，后边文章我们会做介绍。

10.4.7 上采样和下采样
下采样（Downsampling）顾名思义，下采样就是将原始数据“采样”成更小的尺寸，减少数据的分辨率或数量。池化层就是一种常见的对特征图进行下采样的方法。 上采样（Upsampling）上采样就是将原始数据“采样”成更大的尺寸。增加数据的分辨率或者数量，后边我们会介绍对特征图进行上采样的方法。